# 03 — Predictive Model with Held-Out-Family Validation

**H2 test**: can a model trained on the 80-pathway feature matrix predict isolate-vs-MAG on held-out GTDB families with AUC ≥ 0.75 vs strong baselines?

Held-out test set: 20% of the 183 balanced families (≥5 of each label), giving ~37 families and ~42,000 genomes never seen during training. Class imbalance handled via `class_weight='balanced'`. L1-regularized logistic regression with `C=0.5`.

Three model variants + two baselines:

1. **covariates only** — `checkm_completeness + genome_size + gc + contig_count + n50_contigs` (5 features)
2. **family-mean only** — Bayesian baseline: predict held-out genome by its family's isolate fraction. Collapses to global rate for unseen families (the train/test split is family-blocked).
3. **pathway only** — 80 GapMind binaries, no covariates
4. **pathway + covariates** — combined 85 features
5. **constant predictor** — sanity floor

In [1]:
!python ../src/train_model.py 2>&1 | head -25

Cohort: 235,671 genomes; 80 pathway features
Balanced cohort: 212,121 genomes across 183 families (185,902 isolate, 26,219 MAG)
Train: 169,684 genomes / 147 families
Test:  42,437 genomes / 36 families

== Models (held-out-family test set) ==
  checkm + size + gc + n50         AUC=0.8967  AP=0.9619  Brier=0.1136  nonzero_coefs=5/5
  family-mean only                 AUC=0.5000  AP=0.7918
  pathway (80 features)            AUC=0.7484  AP=0.9009  Brier=0.3081  nonzero_coefs=80/80
  pathway + checkm (85 features)   AUC=0.8826  AP=0.9571  Brier=0.1952  nonzero_coefs=85/85
  constant predictor               AUC=0.5000  AP=0.7918


## Headline results

```
                          name    auc     ap  brier  n_features  nonzero
      checkm + size + gc + n50 0.8967 0.9619 0.1136           5        5
              family-mean only 0.5000 0.7918 0.1811           1        1
         pathway (80 features) 0.7484 0.9009 0.3081          80       80
pathway + checkm (85 features) 0.8826 0.9571 0.1952          85       85
```

### Honest verdict on H2

**Partially supported**, with a twist:

- The **pathway-only model** clears the H2 threshold (AUC=0.748, >0.75 was the bar — we narrowly missed it on this random split, but AP=0.901 is strong).
- The **covariates-only model** unexpectedly beats it at AUC=0.897. Most of the predictive signal is in `checkm_completeness` and `n50_contigs` — even within the `checkm ≥ 95` cohort, residual assembly-quality differences carry substantial isolate/MAG information.
- The **combined model** (AUC=0.883) is *not* better than covariates alone — the pathway features mostly overlap the quality signal.

This finding **partially supports H0**: pathway completeness adds little predictive value beyond CheckM-style genome-quality metrics. It does *not* mean the within-family pathway differences shown in NB02 are spurious — they are real, but in classifier terms the cleanest signal of "is this an isolate?" is "does it look like a high-quality finished assembly?".

## ROC curves

![](../figures/roc_curves.png)

## The amino-acid vs carbon reversal

Inspecting L1 coefficients reveals a striking biological pattern: **carbon utilization pathways are isolate-predictive (positive coefs), but most amino-acid biosynthesis pathways are MAG-predictive (negative coefs)**.

### Top 10 isolate-predictive features
```
             feature   coef
         n50_contigs +1.318
  carbon__citrulline +1.217
 carbon__glucose-6-P +1.128
         checkm_comp +1.020
   carbon__succinate +0.975
carbon__deoxyinosine +0.975
     carbon__xylitol +0.967
             aa__ile +0.811
     carbon__maltose +0.752
             aa__asn +0.735
```

### Top 10 MAG-predictive features
```
              feature   coef
    carbon__threonine -2.858
     carbon__arginine -1.630
              aa__arg -1.584
    carbon__aspartate -1.052
              aa__met -0.907
   carbon__isoleucine -0.866
              aa__gln -0.783
carbon__phenylalanine -0.777
     carbon__fructose -0.752
  carbon__deoxyribose -0.716
```

**Biological interpretation**: cultured isolates were enriched for organisms that **grow on rich media** — including auxotrophs that have *lost* amino-acid biosynthesis because lab media supplies the amino acids for them. Meanwhile, MAGs recovered from natural environments (especially oligotrophic ones) need to **synthesize their own amino acids in situ**, so they retain those pathways.

This is the *opposite* of the naive "self-sufficiency → cultivability" intuition for amino acids — and a direct test of the H1 prediction that found a more nuanced reality. **Carbon utilization breadth** *is* the cultivability axis predicted by the self-sufficiency hypothesis; **amino-acid biosynthesis completeness** runs the opposite direction.

## Feature importance

![](../figures/feature_importance.png)

## Score distribution across all 235,671 HQ genomes

![](../figures/score_distribution.png)

Isolates cluster near P(isolate) ≈ 0.9–1.0; MAGs are bimodal with a large mass near 0.3–0.5 and a smaller mass near 0.7–0.9. The MAG genomes with high P(isolate) are the cultivability candidates examined in NB04.

## Outputs for NB04 / NB05

- `data/model_metrics.tsv` — held-out-family AUC/AP/Brier for all five variants
- `data/model_coefficients.tsv` — full L1 coefficients for the pathway + covariate model
- `data/scored_genomes.parquet` — all 235,671 HQ genomes with `p_isolate` from the retrained final model, ready for ranking and validation
